# Lab 48 (solution): Distributed backends and failure handling

Reference implementation. [Lab 46](../../46-scaling-the-signals/) scaled the signals but left three stand-ins: a file lock (needs a shared filesystem), claim-before-send (a failed page keeps its slot), and raw-bytes corpus hashing (a reformat looks like a change). This makes each one production-shaped — a `RedisStore` behind the same `StateStore` interface, a release-on-failure + dead-letter path, and normalized content hashing.

The changes land in the [operating-the-loop](../../41-operating-the-loop/) toolkit (`store.py`, `notify.py`, `canary.py`).

## Step 0: Setup

In [ ]:
import json
import pathlib
import sys
loop = pathlib.Path.cwd().parent / "41-operating-the-loop"
sys.path.insert(0, str(loop))
print("real backends + failure handling + content-aware change in:", loop.name)

## Step 1: A real distributed store (item 1)

`RedisStore` over the same interface; one atomic Lua claim works across serverless and multi-region workers.

In [ ]:
from store import RedisStore, FakeRedis, make_store, InMemoryStore
# Item 1: FileLockStore needed a shared filesystem. Serverless / multi-region workers don't
# share one - they share Redis (or a DB). RedisStore makes the claim a single atomic Lua
# round-trip (cooldown + sliding window), so it works no matter where the workers run.
client = FakeRedis()                      # in production: redis.Redis.from_url(...)
store = RedisStore(client)
print("worker A claims:", store.try_claim("faith::page", now=0.0, cooldown_s=3600, max_per_window=5, window_s=3600))
print("worker B, same incident:", store.try_claim("faith::page", now=10.0, cooldown_s=3600, max_per_window=5, window_s=3600))
# one line picks the backend from config:
print("make_store('memory') ->", type(make_store("memory")).__name__)

## Step 2: Release-on-failure + dead-letter (item 2)

In [ ]:
from notify import deliver, format_alert, to_slack, DeadLetter, post
import os
import tempfile
from urllib.error import URLError
# Item 2: the claim is taken BEFORE delivery, so a send that fails after retries would keep
# the cooldown slot and silence the page. Release-on-failure frees the slot; the dead-letter
# queue records what was missed.
p = format_alert("judged_faithfulness", 0.55, 0.764)
store = RedisStore(FakeRedis())
dlq = DeadLetter(os.path.join(tempfile.mkdtemp(), "dlq.jsonl"))
import notify
orig = notify.post
notify.post = lambda shaped, url: (_ for _ in ()).throw(URLError("endpoint down"))
try:
    status, store = deliver(to_slack(p), p, url="https://hooks.example/x", store=store,
                            now=0.0, sleep=lambda s: None, dead_letter=dlq)
finally:
    notify.post = orig
print("failed delivery ->", status)
print("dead-letter entries:", len(dlq.entries()))
# the slot was freed, so the next run can retry instead of being suppressed
again, store = deliver(to_slack(p), p, url=None, store=store, now=1.0, sleep=lambda s: None)
print("next attempt (slot freed) ->", again)

## Step 3: Normalized content hashing (item 3)

In [ ]:
from canary import normalize_corpus_text, per_doc_fingerprint
import hashlib
# Item 3: the per-document map hashed raw bytes, so a reformat (CRLF, trailing spaces, an
# extra blank line) looked like a content change and triggered canary review. Hash the
# NORMALIZED content instead.
base     = b"# Helix\n\nAanya Rao leads Helix Lab.\n"
reformat = b"# Helix\r\n\n\nAanya Rao leads Helix Lab.   \n\n"   # cosmetic only
changed  = b"# Helix\n\nTomas Vega leads Helix Lab.\n"             # real edit
def h(blob):
    return hashlib.sha256(normalize_corpus_text(blob).encode()).hexdigest()[:12]
print("raw bytes differ on reformat:", hashlib.sha256(base).hexdigest()[:12], "vs", hashlib.sha256(reformat).hexdigest()[:12])
print("normalized hash, base vs reformat:", h(base), "==", h(reformat), "->", h(base)==h(reformat))
print("normalized hash, base vs real edit:", h(base), "!=", h(changed), "->", h(base)!=h(changed))
print("So a reformat triggers no review; a content edit still does.")

## Step 4: The cadence

In [ ]:
# The cadence is unchanged except the nightly notify now dead-letters failed pages
# (and caches the queue), so a flaky webhook never silently drops an alert. The per-document
# review picks up normalized hashing automatically (it is the default).
print("Pick the backend by config (memory | file: | redis://); failed pages are captured,")
print("not lost; and cosmetic corpus edits stop waking the canary review.")

## Step 5: The theme

In [ ]:
# The theme: the stand-ins from Lab 46 become the things you actually run.
#  - a file lock -> Redis/DB (works for serverless and multi-region);
#  - claim-before-send -> release-on-failure + a dead-letter queue (no silent drops);
#  - raw-bytes hashing -> normalized hashing (only content changes count).
print("A stand-in that needs a shared filesystem, loses failed pages, or fires on whitespace")
print("is a stand-in. Swap in the real backend, handle failure, and compare content, not bytes.")

## What you built

The production-shaped versions of Lab 46's three stand-ins: a `RedisStore` whose `try_claim` is a single atomic Lua round-trip (cooldown via a per-key value, rate limit via a global sliding-window sorted set) so the claim holds across serverless and multi-region workers, selectable through `make_store('redis://...')`; a `release` operation on every backend plus a `DeadLetter` queue, so `notify.deliver` frees the cooldown slot and records the payload when a send exhausts its retries instead of silently keeping the slot; and `normalize_corpus_text` so the per-document fingerprint ignores cosmetic reformats (line endings, trailing whitespace, blank-line runs) and fires only on real content changes.

**Where this simplifies:** `FakeRedis` mirrors Redis's single-threaded atomic execution for the demo and tests — production uses a real client and the same Lua; the dead-letter queue is an append-only file (a real one is a durable queue with redelivery and inspection); and normalized hashing is format-insensitive, not meaning-sensitive — a prose reflow that rewraps lines still changes the hash (semantic hashing via embeddings is a separate, noisier tool).

Next: [Lab 49](../../49-graded-gold/) takes the evaluation anchor from binary to graded — an ordinal rubric, a real adjudication protocol, and the Lab 45 annotator weights re-derived against gold.